In [16]:
from azure.core.credentials import AzureKeyCredential
from azure.search.documents.indexes import SearchIndexClient
from azure.search.documents.indexes.models import (
    SearchIndex,
    SimpleField,
    SearchableField,
    SearchFieldDataType,
    SearchField,
    VectorSearch,
    HnswAlgorithmConfiguration,
    VectorSearchProfile
)

from app.core.config import SEARCH_ENDPOINT,SEARCH_ADMIN_KEY

In [17]:
INDEX_NAME = "hybrid-logs-index"

In [18]:
index_client = SearchIndexClient(
    endpoint=SEARCH_ENDPOINT,
    credential=AzureKeyCredential(SEARCH_ADMIN_KEY),
)

fields = [
    SimpleField(name="id", type=SearchFieldDataType.String, key=True),
    SearchableField(name="message", type=SearchFieldDataType.String),
    SearchableField(name="chunk", type=SearchFieldDataType.String),
    SimpleField(name="service_name", type=SearchFieldDataType.String, filterable=True, facetable=True),
    SimpleField(name="log_level", type=SearchFieldDataType.String, filterable=True, facetable=True),
    SimpleField(name="timestamp", type=SearchFieldDataType.DateTimeOffset, filterable=True, sortable=True),
    SearchField(
        name="chunk_vector",
        type=SearchFieldDataType.Collection(SearchFieldDataType.Single),
        searchable=True,
        vector_search_dimensions=1536,
        vector_search_profile_name="my-vector-profile",
    ),
]

vector_search = VectorSearch(
    algorithms=[
        HnswAlgorithmConfiguration(
            name="my-hnsw-config",
        )
    ],
    profiles=[
        VectorSearchProfile(
            name="my-vector-profile",
            algorithm_configuration_name="my-hnsw-config",
        )
    ],
)

index = SearchIndex(name=INDEX_NAME, fields=fields, vector_search=vector_search)

result = index_client.create_or_update_index(index)
print(f"Created or updated index: {result.name}")

Created or updated index: hybrid-logs-index
